***fallback untuk mengambil kembali gambar***

In [4]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

CSV_PATH = "../output/aksesoriAnak_enriched.csv"  # kalau di laptopmu, ganti pathnya
CATEGORY = "fashion-anak-bayi/aksesoris-anak/jepitan-rambut-anak"
SAMPLE_N = 200   # ambil sampel dulu biar cepat (bisa dinaikkan)

df = pd.read_csv(CSV_PATH)

# 1) filter kategori
sub = df[df["category_breadcrumb"] == CATEGORY].copy()
print("Total data kategori:", len(sub))

# 2) normalisasi URL gambar
def fix_img_url(u: str) -> str:
    if not isinstance(u, str):
        return u
    u = u.replace("\\u0026", "&")   # penting!
    if u.startswith("http://"):
        u = "https://" + u[len("http://"):]
    return u

sub["img_fixed"] = sub["mediaURL_image"].astype(str).apply(fix_img_url)

# 3) header khusus untuk akses IMAGE CDN (bukan header API kamu)
IMG_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Referer": "https://www.tokopedia.com/",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

def check_status(url: str) -> int:
    try:
        r = requests.get(url, headers=IMG_HEADERS, timeout=15, stream=True)
        return r.status_code
    except requests.RequestException:
        return -1

# ambil sampel
sample = sub.head(SAMPLE_N).copy()

# 4) cek status paralel biar cepat
statuses = []
with ThreadPoolExecutor(max_workers=20) as ex:
    futs = {ex.submit(check_status, u): u for u in sample["img_fixed"].tolist()}
    for fut in as_completed(futs):
        statuses.append(fut.result())

sample["status"] = statuses

print(sample["status"].value_counts(dropna=False))

# lihat contoh yang gagal
print("\nContoh gagal (top 5):")
print(sample[sample["status"] != 200][["id","name","img_fixed","status"]].head(5))



Total data kategori: 236
status
 403    199
-1        1
Name: count, dtype: int64

Contoh gagal (top 5):
              id                                               name  \
1    12451070010  Shinemi - 14pcs / Set Jepitan Rambut Anak Fash...   
6   100082586008  ByKiddos - Set Aksesories Rambut Anak Isi 18 P...   
8   100087684785  JEPIT RAMBUT ANAK BULU MODEL KELINCI KECIL FAS...   
14  100014628317  BISA COD F6294 Set Aksesoris Rambut Lucu Anak ...   
15  100017435577  Jepitan Rambut Anak Perempuan Lucu Dengan Hias...   

                                            img_fixed  status  
1   https://p16-images-sign-sg.tokopedia-static.ne...     403  
6   https://p16-images-common-sign-sg.tokopedia-st...     403  
8   https://p16-images-common-sign-sg.tokopedia-st...     403  
14  https://p19-images-sign-sg.tokopedia-static.ne...     403  
15  https://p16-images-common-sign-sg.tokopedia-st...     403  


In [5]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

CSV_PATH = "../output/aksesoriAnak_enriched.csv"
CATEGORY = "fashion-anak-bayi/aksesoris-anak/jepitan-rambut-anak"
SAMPLE_N = 20

def fix_img_url(u: str) -> str:
    if not isinstance(u, str):
        return ""
    u = u.replace("\\u0026", "&")
    if u.startswith("http://"):
        u = "https://" + u[len("http://"):]
    return u

IMG_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Referer": "https://www.tokopedia.com/",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

def check_with_session(img_url: str) -> int:
    try:
        s = requests.Session()
        # warm up cookies (penting)
        s.get("https://www.tokopedia.com/", headers={"User-Agent": IMG_HEADERS["User-Agent"]}, timeout=20)

        r = s.get(img_url, headers=IMG_HEADERS, timeout=20, stream=True)
        return r.status_code
    except requests.RequestException:
        return -1

df = pd.read_csv(CSV_PATH)
sub = df[df["category_breadcrumb"] == CATEGORY].copy()
sub["img_fixed"] = sub["mediaURL_image"].astype(str).apply(fix_img_url)

sample = sub.head(SAMPLE_N).copy()

results = []
with ThreadPoolExecutor(max_workers=10) as ex:
    futs = {ex.submit(check_with_session, u): u for u in sample["img_fixed"].tolist()}
    for fut in as_completed(futs):
        results.append(fut.result())

print("Status counts:", pd.Series(results).value_counts())


Status counts: 403    20
Name: count, dtype: int64


In [ ]:
import pandas as pd
import requests, re, os

CSV_PATH = "../output/aksesoriAnak_enriched.csv"
CATEGORY = "fashion-anak-bayi/aksesoris-anak/jepitan-rambut-anak"
LIMIT = 50

OUT_DIR = "img_refresh_test"
os.makedirs(OUT_DIR, exist_ok=True)

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
PAGE_HEADERS = {
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}
IMG_HEADERS = {
    "User-Agent": UA,
    "Referer": "https://www.tokopedia.com/",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

# ambil kandidat url gambar dari HTML
IMG_RE = re.compile(r"https://[^\"'<> ]+tokopedia-static\.net[^\"'<> ]+\.(?:jpg|jpeg|png|webp)[^\"'<> ]*")

def extract_img_candidates(html: str):
    urls = IMG_RE.findall(html)

    # bersihkan + dedup
    seen, out = set(), []
    for u in urls:
        u = u.replace("\\u0026", "&")
        if u not in seen:
            seen.add(u)
            out.append(u)

    # 1) buang aset UI (ikon, logo, dll)
    out = [u for u in out if "assets-tokopedia-lite" not in u]

    # 2) prioritaskan yang terlihat seperti foto produk
    preferred = [u for u in out if "/img/" in u or "tos-alisg-i-aphluv4xwc" in u]
    return preferred if preferred else out


def refresh_and_download(product_url: str, filename: str):
    s = requests.Session()
    # load product page
    r = s.get(product_url, headers=PAGE_HEADERS, timeout=30)
    if r.status_code != 200:
        return None, f"page_{r.status_code}"

    candidates = extract_img_candidates(r.text)
    if not candidates:
        return None, "no_img_found"

    # coba download kandidat satu2 sampai ada yg 200
    for img_url in candidates[:20]:  # batasi biar cepat
        try:
            rr = s.get(img_url, headers=IMG_HEADERS, timeout=30)
            if rr.status_code == 200 and rr.headers.get("content-type","").startswith("image/"):
                path = os.path.join(OUT_DIR, filename)
                with open(path, "wb") as f:
                    f.write(rr.content)
                return img_url, "ok"
        except requests.RequestException:
            continue

    return None, "img_403_or_fail"

df = pd.read_csv(CSV_PATH)
sub = df[df["category_breadcrumb"] == CATEGORY].head(LIMIT).copy()

rows = []
ok = 0
for i, row in sub.iterrows():
    pid = row["id"]
    purl = row["url"]
    img_url, status = refresh_and_download(purl, f"{pid}.jpg")
    rows.append((pid, status, img_url))
    print(pid, status, img_url if img_url else "")
    if status == "ok":
        ok += 1

print("SUCCESS:", ok, "of", len(sub))

result = pd.DataFrame(rows, columns=["id","refresh_status","new_image_url"])
result.to_csv("refresh_result.csv", index=False)
print("Saved: refresh_result.csv")


12451070010 ok https://p16-images-comn-sg.tokopedia-static.net/tos-alisg-i-zr7vqa5nfb-sg/assets-tokopedia-lite/prod/icon144.png~tplv-zr7vqa5nfb-image.image
100082586008 ok https://p16-images-comn-sg.tokopedia-static.net/tos-alisg-i-zr7vqa5nfb-sg/assets-tokopedia-lite/prod/icon144.png~tplv-zr7vqa5nfb-image.image
100087684785 ok https://p16-images-comn-sg.tokopedia-static.net/tos-alisg-i-zr7vqa5nfb-sg/assets-tokopedia-lite/prod/icon144.png~tplv-zr7vqa5nfb-image.image
100014628317 ok https://p16-images-comn-sg.tokopedia-static.net/tos-alisg-i-zr7vqa5nfb-sg/assets-tokopedia-lite/prod/icon144.png~tplv-zr7vqa5nfb-image.image
100017435577 ok https://p16-images-comn-sg.tokopedia-static.net/tos-alisg-i-zr7vqa5nfb-sg/assets-tokopedia-lite/prod/icon144.png~tplv-zr7vqa5nfb-image.image
100004787725 ok https://p16-images-comn-sg.tokopedia-static.net/tos-alisg-i-zr7vqa5nfb-sg/assets-tokopedia-lite/prod/icon144.png~tplv-zr7vqa5nfb-image.image
100031676678 ok https://p16-images-comn-sg.tokopedia-static

## berhasil ambil gambar

In [7]:
import pandas as pd
import requests, re, os

CSV_PATH = "../output/aksesoriAnak_enriched.csv"
CATEGORY = "fashion-anak-bayi/aksesoris-anak/jepitan-rambut-anak"
LIMIT = 50

OUT_DIR = "img_refresh_test"
os.makedirs(OUT_DIR, exist_ok=True)

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
PAGE_HEADERS = {
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}
IMG_HEADERS = {
    "User-Agent": UA,
    "Referer": "https://www.tokopedia.com/",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

# ambil kandidat url gambar dari HTML
IMG_RE = re.compile(r"https://[^\"'<> ]+tokopedia-static\.net[^\"'<> ]+\.(?:jpg|jpeg|png|webp)[^\"'<> ]*")

# kata kunci aset UI yang harus dibuang
UI_BAD_KEYWORDS = [
    "assets-tokopedia-lite", "icon", "logo", "sprite", "favicon",
    "/prod/", "/assets/", "/static/", "gopay", "promo", "rating",
]

def is_ui_asset(url: str) -> bool:
    u = url.lower()
    return any(k in u for k in UI_BAD_KEYWORDS)

def score_product_like(url: str) -> int:
    """
    Skor untuk prioritas foto produk.
    Makin tinggi makin mungkin foto produk.
    """
    u = url.lower()
    score = 0
    if "/img/" in u: score += 5
    if "tos-alisg" in u: score += 3
    if "aphluv4xwc" in u: score += 2
    if "~tplv-" in u: score += 2
    if "white-pad" in u: score += 1
    if "image" in u: score += 1
    # penalti kalau terlihat UI asset
    if is_ui_asset(u): score -= 100
    return score

def extract_img_candidates(html: str):
    urls = IMG_RE.findall(html)

    # bersihkan + dedup
    seen, out = set(), []
    for u in urls:
        u = u.replace("\\u0026", "&")
        if u not in seen:
            seen.add(u)
            out.append(u)

    # buang aset UI
    out = [u for u in out if not is_ui_asset(u)]

    # urutkan berdasarkan skor foto produk (desc)
    out.sort(key=score_product_like, reverse=True)
    return out

def refresh_and_download(product_url: str, filename: str):
    s = requests.Session()

    # load product page
    r = s.get(product_url, headers=PAGE_HEADERS, timeout=30)
    if r.status_code != 200:
        return None, f"page_{r.status_code}"

    candidates = extract_img_candidates(r.text)
    if not candidates:
        return None, "no_img_found"

    # coba download kandidat satu2 sampai dapat foto produk beneran
    for img_url in candidates[:80]:  # naikin sedikit biar ketemu foto produk
        try:
            rr = s.get(img_url, headers=IMG_HEADERS, timeout=30)
            ct = rr.headers.get("content-type", "")
            if rr.status_code == 200 and ct.startswith("image/"):
                # skip gambar kecil (biasanya icon)
                if len(rr.content) < 20_000:  # 20KB
                    continue

                path = os.path.join(OUT_DIR, filename)
                with open(path, "wb") as f:
                    f.write(rr.content)
                return img_url, "ok"
        except requests.RequestException:
            continue

    return None, "img_403_or_fail"

df = pd.read_csv(CSV_PATH)
sub = df[df["category_breadcrumb"] == CATEGORY].head(LIMIT).copy()

rows = []
ok = 0
for _, row in sub.iterrows():
    pid = row["id"]
    purl = row["url"]
    img_url, status = refresh_and_download(purl, f"{pid}.jpg")
    rows.append((pid, status, img_url))
    print(pid, status, img_url if img_url else "")
    if status == "ok":
        ok += 1

print("SUCCESS:", ok, "of", len(sub))

result = pd.DataFrame(rows, columns=["id","refresh_status","new_image_url"])
result.to_csv("refresh_result.csv", index=False)
print("Saved: refresh_result.csv")


12451070010 ok https://p16-images-sign-sg.tokopedia-static.net/tos-alisg-i-aphluv4xwc-sg/img/hDjmkQ/2024/1/2/6bce774a-3e37-4cac-95b5-fcb1fd157982.jpg~tplv-aphluv4xwc-white-pad-v1:1600:1600.jpeg?lk3s=0ccea506&x-expires=1770406470&x-signature=tlLi1D48om1hXWquM6R4K%2F07PvQ%3D&x-signature-webp=AcKmQiClYXyNHk8QEzV%2FD%2BRDFoo%3D
100082586008 ok https://p16-images-sign-sg.tokopedia-static.net/tos-alisg-i-aphluv4xwc-sg/d7a4fd6e367c435884d1a6a30b745776~tplv-aphluv4xwc-resize-jpeg:700:0.jpeg?lk3s=0ccea506&x-expires=1770406473&x-signature=StsVuucq%2B1rb1tpmdb25%2F4B5hMA%3D&x-signature-webp=3UyMYGuwQ%2BT74dmu6AG632ld7rc%3D
100087684785 ok https://p16-images-common-sign-sg.tokopedia-static.net/tos-maliva-i-o3syd03w52-us/581bd1ee220f436794946752db80b107~tplv-o3syd03w52-resize-jpeg:1200:0.jpeg?lk3s=0ccea506&amp;x-expires=1770406476&amp;x-signature=OqLzh7RbsTEbIuzrAOSjz8w7s8s%3D&amp;x-signature-webp=DU5X9ONRrSuclqhIjycjNWceI8s%3D
100014628317 ok https://p16-images-sign-sg.tokopedia-static.net/tos-ali